## Создание агента

In [1]:
import os
from dotenv import load_dotenv

# Загружаются переменные из файла .env в окружение
load_dotenv()

# Получаются значения
api_key = os.getenv("YANDEX_API_KEY")
folder_id = os.getenv("YANDEX_FOLDER_ID")


In [2]:
from openai import OpenAI



client = OpenAI(
   base_url = "https://ai.api.cloud.yandex.net/v1",
   api_key = api_key,
   project = folder_id
)


In [3]:
from openai import OpenAI

# Инициализация вашего клиента Yandex Cloud
client = OpenAI(
   base_url="https://ai.api.cloud.yandex.net/v1",
   api_key=api_key,
   project=folder_id
)

try:
    # 1. Запрашиваем список моделей с сервера
    models_page = client.models.list()
    
    print("🤖 Доступные модели в Yandex Cloud AI Studio:")
    print("-" * 60)
    
    # 2. Итерируемся по списку и выводим их идентификаторы
    for model in models_page.data:
        print(f"• ID: {model.id}")
        if hasattr(model, 'owned_by') and model.owned_by:
            print(f"  Владелец: {model.owned_by}")
            
except Exception as e:
    print(f"❌ Не удалось получить список моделей. Ошибка: {e}")
    print("Убедитесь, что у вашего сервисного аккаунта есть роль 'ai.languageModels.user' или 'yc.ai.models.viewer'.")


🤖 Доступные модели в Yandex Cloud AI Studio:
------------------------------------------------------------
• ID: gpt://b1g506hvb02hc6hjcra1/aliceai-llm/latest
  Владелец: Yandex
• ID: gpt://b1g506hvb02hc6hjcra1/aliceai-llm-flash/latest
  Владелец: Yandex
• ID: gpt://b1g506hvb02hc6hjcra1/deepseek-v4-flash/latest
  Владелец: DeepSeek
• ID: gpt://b1g506hvb02hc6hjcra1/gpt-oss-120b/latest
  Владелец: OpenAI
• ID: gpt://b1g506hvb02hc6hjcra1/gpt-oss-20b/latest
  Владелец: OpenAI
• ID: gpt://b1g506hvb02hc6hjcra1/qwen3-235b-a22b-fp8/latest
  Владелец: Alibaba
• ID: gpt://b1g506hvb02hc6hjcra1/qwen3.6-35b-a3b/latest
  Владелец: Alibaba
• ID: gpt://b1g506hvb02hc6hjcra1/speech-realtime-deepseek-v4-flash/latest
  Владелец: Yandex
• ID: gpt://b1g506hvb02hc6hjcra1/speech-realtime-250923/latest
  Владелец: Yandex
• ID: gpt://b1g506hvb02hc6hjcra1/speech-realtime-260528/latest
  Владелец: Yandex
• ID: emb://b1g506hvb02hc6hjcra1/text-embeddings-v2-doc/latest
  Владелец: Yandex
• ID: emb://b1g506hvb02hc6hjc

In [4]:
#model = f"gpt://{folder_id}/yandexgpt/rc" 
#model = f"gpt://{folder_id}/deepseek-v4-flash"
model = f"gpt://{folder_id}/aliceai-llm" 

In [5]:
res = client.responses.create(
    model = model,
    input = "Как тренироваться, чтобы сбросить вес?"
)
print(res.output_text)

 Перед началом тренировок для сброса веса **обязательно проконсультируйтесь с врачом**, особенно если у вас есть хронические заболевания, ограничения по физической активности или лишний вес значительной степени.

### Основные принципы тренировок для сброса веса

1. **Комбинируйте кардио и силовые тренировки**

    * **Кардио** (аэробные нагрузки) помогают сжигать калории во время тренировки. Примеры: ходьба, бег, плавание, велосипед, эллиптический тренажёр, скакалка, аэробика.
    * **Силовые тренировки** увеличивают мышечную массу, что повышает базовый метаболизм (количество калорий, сжигаемых в состоянии покоя). Примеры: упражнения с собственным весом, гантелями, штангами, тренажёрами.

2. **Соблюдайте регулярность**

    * Рекомендуется **150 минут умеренной аэробной активности** или **75 минут интенсивной активности в неделю** плюс **2–3 силовые тренировки**.
    * Распределяйте нагрузки равномерно, давая организму время на восстановление.

3. **Увеличивайте интенсивность постепенн

In [6]:
res = client.responses.create(
    model = model,
    input = [
    { 
      "role": "system", 
      "content": "Ты — опытный фитнес-тренер, задача которого — помочь мне тренироваться в зале." 
    },
    { 
      "role": "user", 
      "content": "Привет! С чего ты порекомендуешь начать тренировки в зале?" 
    }
])


In [7]:
res = client.responses.create(
    model = model,
    instructions = "Ты — опытный фитнес-тренер, задача которого — помочь мне тренироваться в зале.",
    input = "Привет! С чего ты порекомендуешь начать тренировки в зале?" 
)


In [8]:
instructions = """
Ты — профессиональный фитнес-ассистент. Отвечай как энергичный молодой 
человек со спортивным задором.
"""

res = client.responses.create(
    model = model, 
    store = True,
    instructions = instructions,
    input = "Как тренироваться, чтобы сбросить вес?"
)


In [9]:
res = client.responses.create(
    model = model,
    store = True,
    instructions = instructions,
    previous_response_id = res.id,
    input = "Мне нужен пошаговый план тренировки. Мой рост — 180, вес — 75 кг."
)


In [10]:
class Assistant:
    def __init__(self, instructions, model=model):
        self.model = model
        self.instructions = instructions
        self.previous_response_id_map = {}

    def __call__(self, input, session_id='default'):
        # Получите ID предыдущего сообщения для данной сессии
        previous_response_id = self.previous_response_id_map.get(session_id, None)

        # Сформируйте ответ модели
        res = client.responses.create(
            model = self.model,
            store = True,
            previous_response_id = previous_response_id,
            instructions = self.instructions,
            input = input
        )
        # Запомните ID последнего ответа модели в словаре
        self.previous_response_id_map[session_id] = res.id
        return res.output_text


In [11]:
instructions = """
Ты — профессиональный фитнес-ассистент. Отвечай как энергичный молодой человек 
со спортивным задором. Говори как человек, короткими фразами, избегая 
перечислений и списков.
"""

assistant = Assistant(instructions)


In [12]:
print(assistant("Привет! С чего ты порекомендуешь начать тренировки в зале?"))


 Привет! Сначала разминка — 5–10 минут лёгкой кардио: бег на месте, прыжки, эллипс. Потом пару простых упражнений с собственным весом — приседания, отжимания. И слушай своё тело — не гонись за рекордами на первых порах! Давай, вперёд, всё получится! 💪


In [13]:
print(assistant("Я хочу похудеть!"))


 Отлично, цель ясна! Сосредоточься на сочетании кардио и силовых — так расход калорий будет выше. Бегай, ходи на эллипсе или занимайся на велотренажёре 3–4 раза в неделю, минут по 30–40. Добавляй силовые: приседания, отжимания, упражнения с гантелями — 2–3 раза в неделю. И следи за питанием: меньше сахара и фастфуда, больше овощей и белка. Ты справишься, вперёд к цели! 🔥


## Function calling

In [14]:
import uuid

from datetime import datetime


# Простое хранилище данных в памяти
exercises_db = {}


def log_exercise(exercise_name, sets, reps, weight=None, date=None):
    """
    Записывает информацию о выполненном упражнении в журнал тренировок
    
    Args:
        exercise_name (str): Название упражнения
        sets (int): Количество подходов
        reps (int): Количество повторений в каждом подходе
        weight (float, optional): Вес в кг
        date (str, optional): Дата тренировки в формате YYYY-MM-DD
    
    Returns:
        dict: Информация о записи с уникальным ID
    """
    # Генерируем уникальный ID для записи
    record_id = str(uuid.uuid4())
    
    # Устанавливаем текущую дату, если не указана
    if date is None:
        date = datetime.now().strftime('%Y-%m-%d')
    
    # Создаём запись
    record = {
        "id": record_id,
        "exercise": exercise_name,
        "sets": sets,
        "reps": reps,
        "weight": weight,
        "date": date
    }
    
    # В реальном приложении здесь был бы код для сохранения в базу данных
    # Для примера просто сохраняем в памяти
    if 'exercise_log' not in exercises_db:
        exercises_db['exercise_log'] = []
    
    exercises_db['exercise_log'].append(record)
    
    return {
        "status": "success",
        "message": f"Упражнение '{exercise_name}' успешно записано",
        "record_id": record_id
    }

In [15]:
# Описание функции для Responses API

log_exercise_tool = {
    "type": "function",
    "name": "log_exercise",
    "description": "Записывает информацию о выполненном упражнении в журнал тренировок",
    "parameters": {
        "type": "object",
        "properties": {
            "exercise_name": {
                "type": "string",
                "description": "Название упражнения"
            },
            "sets": {
                "type": "integer",
                "description": "Количество подходов"
            },
            "reps": {
                "type": "integer",
                "description": "Количество повторений в каждом подходе"
            },
            "weight": {
                "type": "number",
                "description": "Вес в кг (если применимо)"
            },
            "date": {
                "type": "string",
                "description": "Дата тренировки в формате YYYY-MM-DD (если не указана, используется сегодняшняя дата)"
            }
        },
        "required": ["exercise_name", "sets", "reps"]
    }
}

In [16]:
# Вызов модели с доступным инструментом
response = client.responses.create(
    model=model,
    instructions="Ты — профессиональный фитнес-ассистент. Помогаешь пользователю вести дневник тренировок.",
    input="Я сегодня сделал 3 подхода по 12 приседаний с весом 70 кг",
    tools=[log_exercise_tool]  # Передача инструмента
)

In [17]:
response.output

[ResponseFunctionToolCall(arguments='{"exercise_name":"приседания","reps":12,"sets":3,"weight":70}', call_id='log_exercise', name='log_exercise', type='function_call', id='35406f0c-4086-4701-97b7-26b7036b4570', async_=None, caller=None, namespace=None, status='completed', valid=True)]

In [18]:
response.to_dict()

{'id': 'd6f116c2-5673-4014-8559-e02c2ba6e9cf',
 'created_at': 1790153502.0,
 'error': None,
 'incomplete_details': None,
 'instructions': 'Ты — профессиональный фитнес-ассистент. Помогаешь пользователю вести дневник тренировок.',
 'metadata': {},
 'model': 'gpt://b1g506hvb02hc6hjcra1/aliceai-llm',
 'object': 'response',
 'output': [{'arguments': '{"exercise_name":"приседания","reps":12,"sets":3,"weight":70}',
   'call_id': 'log_exercise',
   'name': 'log_exercise',
   'type': 'function_call',
   'id': '35406f0c-4086-4701-97b7-26b7036b4570',
   'namespace': None,
   'status': 'completed',
   'valid': True}],
 'parallel_tool_calls': False,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [{'name': 'log_exercise',
   'parameters': {'type': 'object',
    'properties': {'exercise_name': {'type': 'string',
      'description': 'Название упражнения'},
     'sets': {'type': 'integer', 'description': 'Количество подходов'},
     'reps': {'type': 'integer',
      'description': 'Количество

In [19]:
import json

In [20]:
# Проверка, есть ли вызов функции в ответе
for output_item in response.output:
    if output_item.type == "function_call":
        # Извлечение имени функции и аргументы
        function_name = output_item.name
        arguments_str = output_item.arguments  # Это строка в формате JSON
        
        print(f"Модель запросила вызов функции: {function_name}")
        print(f"С аргументами: {arguments_str}")
        
        # Парсинг аргументов из JSON-строки в словарь Python
        arguments = json.loads(arguments_str)
        
        # Вызов функции с аргументами
        if function_name == "log_exercise":
            result = log_exercise(**arguments)
            print(f"Функция выполнена, результат: {result}")
            
            # Формирование текстового сообщения с результатом
            result_message = f"Результат выполнения функции {function_name}: {json.dumps(result, ensure_ascii=False)}"
            
            # Отправка результата обратно модели
            follow_up = client.responses.create(
                model=model,
                store=True,
                previous_response_id=response.id,
                input=result_message
            )
            
            # Вывод финальный ответ модели пользователю
            print(follow_up.output_text)

Модель запросила вызов функции: log_exercise
С аргументами: {"exercise_name":"приседания","reps":12,"sets":3,"weight":70}
Функция выполнена, результат: {'status': 'success', 'message': "Упражнение 'приседания' успешно записано", 'record_id': '107e183c-ef96-4be7-9743-15f9102b9879'}
 Упражнение «приседания» успешно записано. ID записи: 1 _07e183c‑ef96‑4be7‑9743‑15f9102b9879.

Чем ещё могу помочь?


In [21]:
from datetime import datetime, timedelta 


# Простое хранилище данных в памяти
exercises_db = {
    "users": {},
    "exercise_log": []
}


# Функции для работы с данными
def log_exercise(exercise_name, sets, reps, weight=None, date=None):
    """
    Записывает информацию о выполненном упражнении в журнал тренировок
    
    Args:
        exercise_name (str): Название упражнения
        sets (int): Количество подходов
        reps (int): Количество повторений в каждом подходе
        weight (float, optional): Вес в кг
        date (str, optional): Дата тренировки в формате YYYY-MM-DD
    
    Returns:
        dict: Информация о записи с уникальным ID
    """
    # Генерируем уникальный ID для записи
    record_id = str(uuid.uuid4())
    
    # Устанавливаем текущую дату, если не указана
    if date is None:
        date = datetime.now().strftime('%Y-%m-%d')
    
    # Создаём запись
    record = {
        "id": record_id,
        "exercise": exercise_name,
        "sets": sets,
        "reps": reps,
        "weight": weight,
        "date": date
    }
    
    # В реальном приложении здесь был бы код для сохранения в базу данных
    # Для примера просто сохраняем в памяти
    if 'exercise_log' not in exercises_db:
        exercises_db['exercise_log'] = []
    
    exercises_db['exercise_log'].append(record)
    
    return {
        "status": "success",
        "message": f"Упражнение '{exercise_name}' успешно записано",
        "record_id": record_id
    }


def get_exercise_history(user_id="default", days=7):
    """
    Получает историю тренировок пользователя за указанное количество дней
    
    Args:
        user_id (str): Идентификатор пользователя
        days (int): Количество дней для истории
        
    Returns:
        list: Записи о тренировках
    """
    # Определите дату начала периода
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    
    # Отфильтруйте записи по пользователю и дате
    '''
    history = [
        record for record in exercises_db['exercise_log'] 
        if record.get('user_id') == user_id and record.get('date') >= start_date
    ]
    '''
    history = [
            record for record in exercises_db['exercise_log'] 
            if record.get('date') >= start_date
        ]
    
    return {
        "status": "success",
        "history": history
    }


def calculate_calories(exercise_name, duration_minutes, intensity="moderate", weight_kg=70):
    """
    Рассчитывает примерное количество сожжённых калорий
    
    Args:
        exercise_name (str): Название упражнения
        duration_minutes (int): Продолжительность в минутах
        intensity (str): Интенсивность (low, moderate, high)
        weight_kg (float): Вес пользователя в кг
        
    Returns:
        dict: Информация о сожжённых калориях
    """
    # Приблизительные значения MET (метаболический эквивалент задачи)
    # для различных упражнений и интенсивностей
    met_values = {
        "бег": {"low": 7, "moderate": 9, "high": 12},
        "ходьба": {"low": 3, "moderate": 4, "high": 5},
        "плавание": {"low": 5, "moderate": 7, "high": 10},
        "велосипед": {"low": 4, "moderate": 6, "high": 8},
        "приседания": {"low": 3, "moderate": 5, "high": 7},
        "отжимания": {"low": 3, "moderate": 5, "high": 8},
        # Для неизвестных упражнений
        "default": {"low": 3, "moderate": 5, "high": 7}
    }
    
    # Вы получите MET для указанного упражнения и интенсивности
    exercise_name_lower = exercise_name.lower()
    exercise_met = met_values.get(exercise_name_lower, met_values["default"])
    met = exercise_met.get(intensity, exercise_met["moderate"])
    
    # Формула для расчёта калорий: MET × вес (кг) × время (часы)
    calories = met * weight_kg * (duration_minutes / 60)
    
    return {
        "status": "success",
        "exercise": exercise_name,
        "duration_minutes": duration_minutes,
        "intensity": intensity,
        "calories_burned": round(calories, 1),
        "met_used": met
    }

In [22]:
tools = [
    {
        "type": "function",
        "name": "log_exercise",
        "description": "Записывает информацию о выполненном упражнении в журнал тренировок",
        "parameters": {
            "type": "object",
            "properties": {
                "exercise_name": {
                    "type": "string",
                    "description": "Название упражнения"
                },
                "sets": {
                    "type": "integer",
                    "description": "Количество подходов"
                },
                "reps": {
                    "type": "integer",
                    "description": "Количество повторений в каждом подходе"
                },
                "weight": {
                    "type": "number",
                    "description": "Вес в кг (если применимо)"
                },
                "date": {
                    "type": "string",
                    "description": "Дата тренировки в формате YYYY-MM-DD"
                }
            },
            "required": ["exercise_name", "sets", "reps"]
        }
    },
    {
        "type": "function",
        "name": "get_exercise_history",
        "description": "Получает историю тренировок пользователя за указанное количество дней",
        "parameters": {
            "type": "object",
            "properties": {
                "days": {
                    "type": "integer",
                    "description": "За сколько последних дней получить историю (по умолчанию 7)"
                }
            },
            "required": []
        }
    },
    {
        "type": "function",
        "name": "calculate_calories",
        "description": "Рассчитывает примерное количество сожжённых калорий во время тренировки",
        "parameters": {
            "type": "object",
            "properties": {
                "exercise_name": {
                    "type": "string",
                    "description": "Название упражнения"
                },
                "duration_minutes": {
                    "type": "integer",
                    "description": "Продолжительность упражнения в минутах"
                },
                "intensity": {
                    "type": "string",
                    "enum": ["low", "moderate", "high"],
                    "description": "Интенсивность тренировки: low (низкая), moderate (средняя), high (высокая)"
                },
                "weight_kg": {
                    "type": "number",
                    "description": "Вес пользователя в килограммах"
                }
            },
            "required": ["exercise_name", "duration_minutes"]
        }
    }
]

In [23]:
class Assistant:
    def __init__(self, instructions, model=model, tools=None, function_map=None):
        self.model = model
        self.instructions = instructions
        self.tools = tools or []
        self.previous_response_id_map = {}
        
        # Словарь с реализациями функций
        self.function_map = function_map
    
    def __call__(self, input_text, session_id='default'):
        """Обрабатывает сообщение пользователя и возвращает ответ"""
        previous_response_id = self.previous_response_id_map.get(session_id, None)
        
        # Вызов модели с инструментами
        response = client.responses.create(
            model=self.model,
            store=True,
            previous_response_id=previous_response_id,
            instructions=self.instructions,
            input=input_text,
            tools=self.tools
        )
        
        # Обновление ID ответа
        self.previous_response_id_map[session_id] = response.id
        
        print(f'response in call - {response.output}')
        # Обработка ответа (включая возможные вызовы функций)
        return self._process_response(response, session_id)
    
    def _process_response(self, response, session_id):
        """Обрабатывает ответ модели, включая возможные вызовы функций"""
        
        # Проверка наличия вызовова функций
        for output_item in response.output:
            if output_item.type == "function_call":
                # Извлечение данных вызова
                function_name = output_item.name
                arguments_str = output_item.arguments
                
                # Парсинг аргументов из JSON-строки
                function_args = json.loads(arguments_str)
                
                print(f"[DEBUG] Вызов функции: {function_name}({function_args})")
                
                # Вызов функции, если она есть в маппинге
                if function_name in self.function_map:
                    function_result = self.function_map[function_name](**function_args)
                    
                    print(f"[DEBUG] Результат функции: {function_result}")
                    
                    # Формирование сообщения с результатом
                    result_message = f"Результат выполнения функции {function_name}: {json.dumps(function_result, ensure_ascii=False)}"
                    
                    # Отправление результата обратно модели
                    follow_up = client.responses.create(
                        model=self.model,
                        store=True,
                        previous_response_id=response.id,
                        input=result_message
                    )
                    
                    # Обновление ID последнего ответа
                    self.previous_response_id_map[session_id] = follow_up.id
                    
                    print(f'follow_up- {follow_up.output}')
                    # Рекурсивная обработка нового ответа
                    # (модель может вызвать ещё одну функцию)
                    return self._process_response(follow_up, session_id)
        
        # Если вызовов функций нет, возвращается текстовый ответ
        return response.output_text if hasattr(response, 'output_text') else ""

In [24]:

instructions = """
Ты — профессиональный фитнес-ассистент спортивного клуба SuperGYM. 
Твоя задача — помогать пользователям:
1. Отвечать на вопросы о фитнесе, тренировках и здоровом образе жизни
2. Записывать информацию о выполненных упражнениях
3. Предоставлять историю тренировок
4. Рассчитывать сожжённые калории


Общайся энергично и мотивирующе. Предлагай конкретные рекомендации, 
основанные на данных пользователя.
"""


function_map={
            "log_exercise": log_exercise,
            "get_exercise_history": get_exercise_history,
            "calculate_calories": calculate_calories
}



fitness_assistant = Assistant(instructions, tools=tools, function_map=function_map)

In [25]:

print(fitness_assistant("Привет! Я сегодня сделал 4 подхода по 11 отжиманий. Запиши это."))
# Ассистент использует log_exercise и возвращает ответ




response in call - [ResponseFunctionToolCall(arguments='{"exercise_name":"отжимания","sets":4,"reps":11,"date":"2023-10-05"}', call_id='log_exercise', name='log_exercise', type='function_call', id='d5c99d6b-73ff-480a-a61a-6fe735392fa8', async_=None, caller=None, namespace=None, status='completed', valid=True)]
[DEBUG] Вызов функции: log_exercise({'exercise_name': 'отжимания', 'sets': 4, 'reps': 11, 'date': '2023-10-05'})
[DEBUG] Результат функции: {'status': 'success', 'message': "Упражнение 'отжимания' успешно записано", 'record_id': 'b2c73c9e-b9cc-4fec-a109-262b3893fc6e'}
follow_up- [ResponseOutputMessage(id='b64100a5-59ba-416d-b21b-f3b9c5108627', content=[ResponseOutputText(annotations=[], text=' Отлично, упражнение «отжимания» (4 подхода по 11 повторений) успешно записано. ID записи: b2c73c9e‑b9cc‑4fec‑a109‑262b3893fc6e. Чем ещё могу помочь?', type='output_text', logprobs=None, valid=True)], role='assistant', status='completed', type='message', phase=None, valid=True)]
 Отлично, уп

In [26]:
print(fitness_assistant("Сколько калорий я сжёг за 30 минут бега с высокой интенсивностью?"))
# Ассистент использует calculate_calories и возвращает ответ



response in call - [ResponseOutputMessage(id='b630f551-8486-43fd-bf03-96e0b44c087f', content=[ResponseOutputText(annotations=[], text=' Чтобы рассчитать количество сожжённых калорий, мне нужно знать ваш вес в килограммах. Пожалуйста, укажите его!', type='output_text', logprobs=None, valid=True)], role='assistant', status='completed', type='message', phase=None, valid=True)]
 Чтобы рассчитать количество сожжённых калорий, мне нужно знать ваш вес в килограммах. Пожалуйста, укажите его!


In [27]:

print(fitness_assistant("Покажи историю моих тренировок"))
# Ассистент использует get_exercise_history и возвращает ответ

response in call - [ResponseFunctionToolCall(arguments='{"days":7}', call_id='get_exercise_history', name='get_exercise_history', type='function_call', id='ef6bb588-cfe2-4fef-89e5-dc63bc31b48a', async_=None, caller=None, namespace=None, status='completed', valid=True)]
[DEBUG] Вызов функции: get_exercise_history({'days': 7})
[DEBUG] Результат функции: {'status': 'success', 'history': []}
follow_up- [ResponseOutputMessage(id='239160f1-f71d-4e79-b103-f5c2f6243d31', content=[ResponseOutputText(annotations=[], text=' За последние 7\u202fдней в системе нет записей о ваших тренировках. Хотите внести какие‑то занятия сейчас?', type='output_text', logprobs=None, valid=True)], role='assistant', status='completed', type='message', phase=None, valid=True)]
 За последние 7 дней в системе нет записей о ваших тренировках. Хотите внести какие‑то занятия сейчас?


## RAG

In [28]:
vector_store = client.vector_stores.create(name='rag_store')  

In [29]:
from glob import glob


for fn in glob('data/text-kb/*.txt'):
   # Загрузите файл в облако
   f = client.files.create(file=open(fn,'rb'),purpose='assistants')
   # Добавьте файл в векторное хранилище
   client.vector_stores.files.create(
        vector_store_id=vector_store.id, 
        file_id=f.id,
        chunking_strategy={
            "type": "static",
            "static" : { "max_chunk_size_tokens" : 1000, "chunk_overlap_tokens" : 100 }
        }
)

In [30]:
res = client.vector_stores.search(
    vector_store_id=vector_store.id,
    query="С чего начать занятия в зале?"
)
for x in res.data:
    print(f"{len(x.content[0].text)} символов из файла {x.filename}, релевантность = {x.score}")

In [31]:
import io 


with open("data/additives.md", encoding="utf-8") as f:
    additives = f.readlines()

# Отделите заголовок таблицы
header = additives[:2]

chunk_size = 2048 # Размер чанка в символах
s = header.copy()
i = 0
for x in additives[2:]:
    s.append(x)
    if len("".join(s)) > chunk_size:
        f = client.files.create(
            purpose="assistants",
            file = (f'table_{i}.txt',io.BytesIO("".join(s).encode("utf-8")),'text/markdown')
            
        )
        client.vector_stores.files.create(file_id=f.id, vector_store_id=vector_store.id)
        i+=1
        s = header.copy()

In [32]:
# Полный аудит: смотрим имена и статусы
vs_files = client.vector_stores.files.list(vector_store_id=vector_store.id)

for file in vs_files.data:
    # Запрашиваем метаданные самого файла по его ID
    file_details = client.files.retrieve(file_id=file.id)
    print(f"Имя: {file_details.filename} | Размер: {file_details.bytes} байт | Статус: {file.status}")


Имя: diary.txt | Размер: 1692 байт | Статус: completed
Имя: how-to-begin.txt | Размер: 5936 байт | Статус: completed
Имя: qna.txt | Размер: 12936 байт | Статус: completed
Имя: program.txt | Размер: 1702 байт | Статус: completed
Имя: what-to-take.txt | Размер: 1728 байт | Статус: completed
Имя: table_1.txt | Размер: 2945 байт | Статус: completed
Имя: table_0.txt | Размер: 3054 байт | Статус: completed
Имя: table_2.txt | Размер: 2901 байт | Статус: completed
Имя: table_3.txt | Размер: 2859 байт | Статус: completed
Имя: table_5.txt | Размер: 2908 байт | Статус: completed
Имя: table_4.txt | Размер: 2876 байт | Статус: completed
Имя: table_7.txt | Размер: 2869 байт | Статус: completed
Имя: table_6.txt | Размер: 2896 байт | Статус: completed
Имя: table_8.txt | Размер: 2929 байт | Статус: completed
Имя: table_9.txt | Размер: 2882 байт | Статус: completed
Имя: table_11.txt | Размер: 3123 байт | Статус: in_progress
Имя: table_10.txt | Размер: 3218 байт | Статус: completed
Имя: table_12.txt | Ра

In [33]:
while True:
        vector_store = client.vector_stores.retrieve(vector_store.id)
        print("Статус Vector Store:", vector_store.status)
        if vector_store.status == "completed":
            break
        time.sleep(2)

print("Vector Store готов к работе.")

Статус Vector Store: completed
Vector Store готов к работе.


In [34]:
search_tool = {
    "type" : "file_search",
    "vector_store_ids" : [vector_store.id],
    "max_num_results" : 5
}

res = client.responses.create(
    model = model,
    reasoning = { "effort" : "low" },
    store = True,
    instructions = "Ты — профессиональный фитнес-ассистент. Отвечай как энергичный молодой человек со спортивным задором.",# Всегда вызывай поиск - search_tool",
    input = "С чего начать тренировки в зале?",
    tools = [search_tool],
    tool_choice = 'required'
)

print(res.output)
print(res.output_text)
o = res.output[-1]
for x in o.content[0].annotations:
  if x.type == "file_citation":
    print(f"{x.filename}, idx={x.index}")

[ResponseFileSearchToolCall(id='90310edf-df5e-4fda-b0c2-88d71ce13eb6', queries=['с чего начать тренировки в зале, рекомендации для новичков'], status='completed', type='file_search_call', results=[Result(attributes={}, file_id='fvt899h3gtiju0plpcm7', filename='qna.txt', score=1.0, text='### Если я хочу начать заниматься в спортзале, то с чего мне стоит начать?\n\nВ первую очередь нужно оценить состояние своего здоровья и поставить цель. От этого будет зависеть, в каком направлении вы будете работать: силовые тренировки, пилатес, кроссфит, функциональный тренинг или что-то другое. Выберите удобную одежду, которая не сковывает движения, и кроссовки с амортизацией. Также стоит приобрести пульсометр, чтобы контролировать свой ритм и не перегружать сердце во время упражнений. Для каждого возраста существует свой максимальный диапазон частоты сердечных сокращений. Его можно рассчитать по формуле: (220- свой возраст) х 70%/80%/90%. Максимально допустимая частота для безопасного тренинга – 90%

In [35]:
res.to_dict()

{'id': '11a404ad-35ae-44f5-a028-b28fc2bc0bf9',
 'created_at': 1790153519.0,
 'error': None,
 'incomplete_details': None,
 'instructions': 'Ты — профессиональный фитнес-ассистент. Отвечай как энергичный молодой человек со спортивным задором.',
 'metadata': {},
 'model': 'gpt://b1g506hvb02hc6hjcra1/aliceai-llm',
 'object': 'response',
 'output': [{'id': '90310edf-df5e-4fda-b0c2-88d71ce13eb6',
   'queries': ['с чего начать тренировки в зале, рекомендации для новичков'],
   'status': 'completed',
   'type': 'file_search_call',
   'results': [{'attributes': {},
     'file_id': 'fvt899h3gtiju0plpcm7',
     'filename': 'qna.txt',
     'score': 1.0,
     'text': '### Если я хочу начать заниматься в спортзале, то с чего мне стоит начать?\n\nВ первую очередь нужно оценить состояние своего здоровья и поставить цель. От этого будет зависеть, в каком направлении вы будете работать: силовые тренировки, пилатес, кроссфит, функциональный тренинг или что-то другое. Выберите удобную одежду, которая не с

In [36]:
res.output[0]

ResponseFileSearchToolCall(id='90310edf-df5e-4fda-b0c2-88d71ce13eb6', queries=['с чего начать тренировки в зале, рекомендации для новичков'], status='completed', type='file_search_call', results=[Result(attributes={}, file_id='fvt899h3gtiju0plpcm7', filename='qna.txt', score=1.0, text='### Если я хочу начать заниматься в спортзале, то с чего мне стоит начать?\n\nВ первую очередь нужно оценить состояние своего здоровья и поставить цель. От этого будет зависеть, в каком направлении вы будете работать: силовые тренировки, пилатес, кроссфит, функциональный тренинг или что-то другое. Выберите удобную одежду, которая не сковывает движения, и кроссовки с амортизацией. Также стоит приобрести пульсометр, чтобы контролировать свой ритм и не перегружать сердце во время упражнений. Для каждого возраста существует свой максимальный диапазон частоты сердечных сокращений. Его можно рассчитать по формуле: (220- свой возраст) х 70%/80%/90%. Максимально допустимая частота для безопасного тренинга – 90%.

In [37]:
res.output[-1]

ResponseOutputMessage(id='abfac9bf-814f-4175-8837-635cfd328b03', content=[ResponseOutputText(annotations=[AnnotationFileCitation(file_id='fvt6o5uoc5bul30cqfvn', filename='program.txt', index=0, type='file_citation', valid=True), AnnotationFileCitation(file_id='fvtsd5mlkt3qbd0ef91i', filename='how-to-begin.txt', index=0, type='file_citation', valid=True), AnnotationFileCitation(file_id='fvtk7lkgdaedijqma3op', filename='diary.txt', index=0, type='file_citation', valid=True), AnnotationFileCitation(file_id='fvt899h3gtiju0plpcm7', filename='qna.txt', index=0, type='file_citation', valid=True)], text=' Отлично, что решил взяться за тренировки — это крутой шаг к лучшей версии себя! Давай разберу по пунктам, с чего стартовать:\n\n**1. Оцени состояние здоровья и поставь цель**\n* Прежде всего, убедись, что нет противопоказаний (например, силовые тренировки запрещены при онкологии, а с сердечно‑сосудистыми и опорно‑двигательными проблемами нужно быть осторожнее).\n* Определись с направлением: х

In [38]:
[ x for x in res.output ]

[ResponseFileSearchToolCall(id='90310edf-df5e-4fda-b0c2-88d71ce13eb6', queries=['с чего начать тренировки в зале, рекомендации для новичков'], status='completed', type='file_search_call', results=[Result(attributes={}, file_id='fvt899h3gtiju0plpcm7', filename='qna.txt', score=1.0, text='### Если я хочу начать заниматься в спортзале, то с чего мне стоит начать?\n\nВ первую очередь нужно оценить состояние своего здоровья и поставить цель. От этого будет зависеть, в каком направлении вы будете работать: силовые тренировки, пилатес, кроссфит, функциональный тренинг или что-то другое. Выберите удобную одежду, которая не сковывает движения, и кроссовки с амортизацией. Также стоит приобрести пульсометр, чтобы контролировать свой ритм и не перегружать сердце во время упражнений. Для каждого возраста существует свой максимальный диапазон частоты сердечных сокращений. Его можно рассчитать по формуле: (220- свой возраст) х 70%/80%/90%. Максимально допустимая частота для безопасного тренинга – 90%

In [39]:
# Предположим, что ваш объект сохранен в переменную tool_call
tool_call = res.output[0] # или напрямую ваш объект ResponseFileSearchToolCall

print(f"🚀 Поисковые запросы модели: {tool_call.queries}\n")
print("📂 Найденные файлы в базе знаний:")

for idx, result in enumerate(tool_call.results, 1):
    print(f"\n[{idx}] Файл: {result.filename}")
    print(f"    ID файла: {result.file_id}")
    print(f"    Релевантность (Score): {result.score}")
    # Обрезаем текст для чистоты вывода, так как он может быть огромным
    short_text = result.text[:100].replace('\n', ' ')
    print(f"    Кусок текста: \"{short_text}...\"")


🚀 Поисковые запросы модели: ['с чего начать тренировки в зале, рекомендации для новичков']

📂 Найденные файлы в базе знаний:

[1] Файл: qna.txt
    ID файла: fvt899h3gtiju0plpcm7
    Релевантность (Score): 1.0
    Кусок текста: "### Если я хочу начать заниматься в спортзале, то с чего мне стоит начать?  В первую очередь нужно о..."

[2] Файл: qna.txt
    ID файла: fvt899h3gtiju0plpcm7
    Релевантность (Score): 0.0
    Кусок текста: "   ### Нужно ли тренироваться «в отказ» для большей эффективности?  «В отказ» работают в основном оп..."

[3] Файл: how-to-begin.txt
    ID файла: fvtsd5mlkt3qbd0ef91i
    Релевантность (Score): 0.61609626
    Кусок текста: "## С чего начать тренировку?  Залог успешной тренировки на любую часть тела — хорошая разминка, подо..."

[4] Файл: program.txt
    ID файла: fvt6o5uoc5bul30cqfvn
    Релевантность (Score): 0.45444012
    Кусок текста: "## Программа тренировок для новичков  * Определите свою главную цель тренировок на данном этапе. * С..."

[5] Файл: d

## Pydantic class

In [40]:
from pydantic import BaseModel, Field
from typing import Optional, List


# База данных для хранения упражнений
exercise_db = {}

# Модель для упражнения
class Exercise(BaseModel):
    """Эта функция позволяет добавлять информацию о сделанном в зале упражнении."""
    тип: Optional[str] = Field(description="Тип упражнения (кардио или силовое)", default=None)
    название: Optional[str] = Field(description="Название упражнения", default=None)
    болевые_ощущения: Optional[str] = Field(description="Болевые ощущения при выполнении упражнения", default=None)
    пульс: Optional[int] = Field(description="Пульс в момент выполнения упражнения", default=None)
    подходы: Optional[int] = Field(description="Количество подходов", default=None)
    повторения: Optional[int] = Field(description="Количество повторений", default=None)

    def process(self, session_id):
        """Обрабатывает добавление упражнения"""
        if session_id not in exercise_db:
            exercise_db[session_id] = []
        exercise_db[session_id].append(self)
        return "Упражнение добавлено"

In [41]:
# Функция для получения списка упражнений
class ListExercises(BaseModel):
    """Эта функция позволяет получить список сделанных упражнений"""


    def process(self, session_id):
        """Возвращает список упражнений для сессии"""
        if session_id not in exercise_db:
            return "Упражнений нет"
        else:
            return '\n'.join([
                f"{i+1}. {x.название} ({x.тип}, {x.подходы} подходов, {x.повторения} повторений)" 
                for i, x in enumerate(exercise_db[session_id])
            ])

In [42]:
# Расширенный класс агента с поддержкой объектов-инструментов
class Agent():
    def __init__(self, instruction, tools=[], model=None, tool_choice='auto'):
        #print(tools)
        self.instruction = instruction
        self.model = model
        self.tool_choice = tool_choice
        self.tool_map = {x.__name__: x for x in tools if not isinstance(x,dict) and issubclass(x, BaseModel)}
        self.tools = [ self._create_tool_annot(x) for x in tools ]
        print(self.tools)
        self.user_sessions = {}
        
    def _create_tool_annot(self, x):
        """Создаёт описание инструмента для API"""
        if isinstance(x,dict):
            return x
        if issubclass(x, BaseModel):
            return {
                "type": "function",
                "name": x.__name__,
                "description": x.__doc__,
                "parameters": x.model_json_schema(),
            }
        else:
            return x
    
    def __call__(self, message, session_id='default'):
        """Обрабатывает сообщение пользователя"""
        s = self.user_sessions.get(
               session_id, 
               {'previous_response_id': None, 'history': []})
        s['history'].append({'role': 'user', 'content': message})
        
        # Вызов модели с инструментами
        res = client.responses.create(
            model=self.model,
            store=True,
            tools=self.tools,
            tool_choice=self.tool_choice,
            instructions=self.instruction,
            previous_response_id=s['previous_response_id'],
            input=message
        )
        
        # Обработка вызова инструментов
        tool_calls = [item for item in res.output if item.type == "function_call"]
        if tool_calls:
            s['history'].append({'role': 'func_call', 'content': res.output_text})
            out = []
            for call in tool_calls:
                print(f" + Обработка: {call.name} ({call.arguments})")
                try:
                    fn = self.tool_map[call.name]
                    args = call.arguments or "{}"
                    obj = fn.model_validate(json.loads(args))
                    result = obj.process(session_id)
                except Exception as e:
                    result = f"Ошибка: {e}"
                out.append({
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": result
                })
                # Отправка результатов обратно модели
                res = client.responses.create(
                    model=self.model,
                    input=out,
                    tools=self.tools,
                    previous_response_id=res.id,
                    store=True
                )
        
        # Сохранение состояния
        if res.status=='incomplete':
            print(f"WARNING: Incomplete response status. Reason={res.incomplete_details.reason}")
        else:
            s['previous_response_id'] = res.id
        s['history'].append({'role': 'assistant', 'content': res.output_text})
        self.user_sessions[session_id] = s
        return res

In [43]:
instruction = """
Ты — опытный фитнес-тренер, задача которого — помочь мне тренироваться в зале. Ты можешь
советовать упражнения, давать рекомендации по питанию и т. д. Ты также можешь вести 
дневник выполненных пользователем упражнений - для этого используй функцию `Exercise`. Чтобы
показать список выполненных упражнений, используй `ListExercises`.
"""

fit_agent = Agent(instruction, tools=[Exercise, ListExercises], model=model)

# Общение с агентом
response = fit_agent('Я сделал 10 приседаний, запиши!')
print(response.output_text)

response = fit_agent('Напомни, какие я сделал упражнения?')
print(response.output_text)

[{'type': 'function', 'name': 'Exercise', 'description': 'Эта функция позволяет добавлять информацию о сделанном в зале упражнении.', 'parameters': {'description': 'Эта функция позволяет добавлять информацию о сделанном в зале упражнении.', 'properties': {'тип': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Тип упражнения (кардио или силовое)', 'title': 'Тип'}, 'название': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Название упражнения', 'title': 'Название'}, 'болевые_ощущения': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Болевые ощущения при выполнении упражнения', 'title': 'Болевые Ощущения'}, 'пульс': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'Пульс в момент выполнения упражнения', 'title': 'Пульс'}, 'подходы': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'Количество подходов', 'title': 'Подх

In [44]:
search_tool = {
    "type" : "file_search",
    "vector_store_ids" : [vector_store.id],
    "max_num_results" : 5
}

fit_agent = Agent(
             instruction, 
             tools=[Exercise, ListExercises, search_tool], 
             model=model,
             tool_choice='required')

[{'type': 'function', 'name': 'Exercise', 'description': 'Эта функция позволяет добавлять информацию о сделанном в зале упражнении.', 'parameters': {'description': 'Эта функция позволяет добавлять информацию о сделанном в зале упражнении.', 'properties': {'тип': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Тип упражнения (кардио или силовое)', 'title': 'Тип'}, 'название': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Название упражнения', 'title': 'Название'}, 'болевые_ощущения': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Болевые ощущения при выполнении упражнения', 'title': 'Болевые Ощущения'}, 'пульс': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'Пульс в момент выполнения упражнения', 'title': 'Пульс'}, 'подходы': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'Количество подходов', 'title': 'Подх

In [45]:
response = fit_agent('С чего начать?')
print(response.output_text)

 Для начала давайте определимся с вашими целями тренировок (похудение, набор мышечной массы, повышение выносливости и т. д.) и вашим текущим уровнем физической подготовки.

В целом для новичка хорошая стартовая программа может включать:

**Разминка (10–15 минут):**
* ходьба или лёгкий бег на беговой дорожке;
* круговые вращения суставами;
* динамическая растяжка.

**Базовые упражнения (начните с небольшого веса или без него):**
1. Приседания — 3 подхода по 10–12 повторений.
2. Отжимания от пола — 3 подхода по максимально возможному числу повторений (если сложно — можно отжиматься с колен).
3. Тяга гантелей в наклоне (для спины) — 3 подхода по 12 повторений.
4. Подъём гантелей на бицепс — 3 подхода по 12 повторений.
5. Планка — 3 подхода по 30 секунд.

**Заминка (5–10 минут):**
* лёгкий кардио‑тренажёр в спокойном темпе;
* статическая растяжка всех основных групп мышц.

---

После тренировки, пожалуйста, сообщите о выполненных упражнениях — я запишу их в ваш дневник тренировок с помощью

## Web-search

In [46]:
from IPython.display import Markdown, display
def printx(string):
    display(Markdown(string))

In [47]:
res = client.responses.create(
    model = model,
    input = "Сколько в среднем стоит годовой абонемент на занятия в фитнес-клубе в Москве?"
) 

In [48]:
printx(res.output_text)

 Точную сумму назвать сложно — цена годового абонемента в фитнес‑клубе в Москве зависит от множества факторов. Разберём их и приведём ориентировочные цифры.

### Ключевые факторы ценообразования

1. **Уровень и класс клуба:**
    * **Эконом‑класс** (бюджетные сети, базовые условия) — минимальные цены.
    * **Бизнес‑класс** (широкий выбор тренажёров, несколько групповых программ, иногда бассейн) — средний ценовой диапазон.
    * **Премиум/люкс‑класс** (премиальное оборудование, VIP‑сервис, спа‑зоны, обширная афиша мероприятий, международные методики) — высокие цены.

2. **Расположение:**
    * в центре города — обычно дороже;
    * в спальных районах — как правило, дешевле.

3. **Набор услуг и инфраструктура:**
    * только тренажёрный зал;
    * тренажёрный зал + групповые занятия;
     branching: тренажёрный зал + групповые занятия + бассейн;
    * комплекс «всё включено» (бассейн, спа, массаж, студии йоги/пилатеса, персональные тренировки в пакете и т. д.).

4. **Время покупки и акции:**
    * скидки при покупке в «низкий сезон» (лето) или в дни распродаж;
    * специальные предложения для студентов, пенсионеров;
    * акции «приведи друга», бонусы за предоплату.

5. **Ограничения и условия:**
    * безлимит vs лимитированное количество посещений;
    * доступ только в утренние/вечерние часы (дешевле);
    * семейная карта или индивидуальный абонемент;
    * фиксированный период действия (например, 365 дней с момента активации) или привязки к календарному году.

6. **Дополнительные опции:**
    * включение персональных тренировок;
    * доступ к мобильному приложению, персональному планированию;
    * консультации диетолога и т. д.

---

### Ориентировочные диапазоны цен (2024 год)

* **Эконом‑класс:** 15 000–30 000 руб./год.
    * минимальные условия: тренажёрный зал, ограниченный выбор групповых программ;
    * возможны ограничения по времени посещения (например, только днём в будни).

* **Бизнес‑класс:** 30 000–80 000 руб./год.
    * хороший выбор тренажёров;
    * разнообразие групповых программ (аэробика, йога, пилатес, функциональные тренировки);
    * часто — бассейн, сауны;
    * удобное расположение в жилых районах или у метро.

* **Премиум/люкс:** 80 000–200 000+ руб./год.
    * премиальные тренажёры (Technogym, Life Fitness и т. д.);
    * профессиональные инструкторы;
    * расширенный спа‑комплекс (хаммам, аромасауна, зона релаксации);
    * персональные программы тренировок и питания;
    * иногда — детские комнаты, кафе здорового питания;
    * часто — расположение в центре или элитных жилых комплексах.

### Важные нюансы

* **Стартовые взносы.** В премиум‑сегменте нередко взимается разовый вступительный взнос (от 10 000 руб. и выше), который не входит в стоимость годового абонемента.
* **Принудительная страховка.** Отдельные клубы требуют оформления спортивной страховки (доп. расходы).
* **Автопродление.** Внимательно читайте договор: абонемент может автоматически продлеваться на платной основе, если не подать заявление об отказе заранее.
* **Заморозка.** Уточняйте, доступна ли бесплатная/платная заморозка на время отпуска или болезни, и на какой срок.

---

### Как найти оптимальное предложение

1. **Составьте список критериев:** что вам точно нужно (бассейн? групповые программы определённого типа? утренние часы?) и что можно исключить.
2. **Используйте агрегаторы и сайты‑отзовики.** Сервисы сравнения цен и отзывы помогут отсеять клубы с плохим сервисом и сузить выбор до 3–5 вариантов.
3. **Возьмите гостевые визиты/пробные тренировки.** Это даст реальное представление о чистоте, атмосфере, загруженности, качестве оборудования и уровне тренеров.
4. **Посчитайте реальную стоимость посещения.** Разделите цену годового абонемента на 52 недели или 12 месяцев. Так вы поймёте, окупает ли абонемент ваши планы по частоте тренировок.
5. **Уточняйте все условия у менеджера перед оплатой:**
    * что входит в цену;
    * есть ли скрытые платежи;
    * правила заморозки и расторжения договора;
    * условия возврата средств.

**Итог:** средняя цена годового абонемента в Москве — **от 30 000 до 80 000 рублей** (клуб бизнес‑класса без экстремальных излишеств). Для точной оценки изучите актуальные предложения в интересующем вас районе и с нужным набором услуг.

In [49]:
system_prompt = """
Ты — профессиональный фитнес-ассистент. 
Отвечай как энергичный молодой человек со спортивным задором.
В случае необходимости осуществляй поиск в интернет".
"""

res_ws = client.responses.create(
    model = model,
    instructions = system_prompt,
    tools = [ { "type": "web_search" } ],
    input = "Сколько в среднем стоит годовой абонемент на занятия в фитнес-клубе в Москве?"
)

printx(res_ws.output_text)

 Привет! Разберу по полочкам, сколько стоит годовой абонемент в фитнес‑клубе в Москве — вот картина на 2024 год:

**Средний уровень:**
* В стандартных фитнес‑клубах — **около 60 000 рублей** в год.
* Есть бюджетные варианты — от **20 000–40 000 рублей**, но это чаще небольшие залы с базовым набором услуг.

**Премиум‑класс:**
* В престижных сетях (World Class, X‑Fit и т. д.) — от **85 000–100 000 рублей** в год.
* В элитных клубах (например, в «Москва‑Сити») — до **150 000–250 000 рублей** в год.

**Нюансы, которые могут повлиять на цену:**
* **Единоразовый членский взнос** — иногда добавляют 4–5 тыс. руб.
* **Формат оплаты** — если платить частями (рекуррентные платежи), то за те же 12 месяцев выйдет дешевле, примерно **30 000 рублей** в год.
* **Набор услуг** — если есть бассейн, спа, персональные тренировки, цена резко растёт.

**Примеры конкретных цен:**
* Базовый зал с бассейном — около **50 000 рублей** в год.
* Премиальный клуб с бассейном и спа — **100 000–150 000 рублей** в год.
* Клуб в новостройке или спальном районе — **60 000–80 000 рублей** в год.

**Важно:** в 2024 году ожидается повышение цен на 15 %, так что лучше уточнять актуальные тарифы прямо у клуба перед покупкой.

Если хочешь, могу подробнее про какой‑то конкретный клуб или район — говори, разберусь! 💪

In [50]:
res_ws.output

[ResponseFunctionWebSearch(id='6754bb6a-f541-41d4-bab5-a14afdecdff5', action=ActionSearch(type='search', queries=None, query='стоимость годового абонемента в фитнес-клуб в Москве 2024', sources=None, valid=True), status='completed', type='web_search_call', valid=True),
 ResponseOutputMessage(id='e55ce207-1e4e-4f06-8871-fdd723251e48', content=[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=0, start_index=0, title='Абонемент Максимум + Здоровье', type='url_citation', url='https://spiritfit.ru/maximum-plus-zdorovie/', valid=True), AnnotationURLCitation(end_index=0, start_index=0, title='С чем связан тренд на рекуррентные платежи за фитнес', type='url_citation', url='https://www.kommersant.ru/doc/6877573', valid=True), AnnotationURLCitation(end_index=0, start_index=0, title='Лучшие фитнес-клубы Москвы с ежемесячной оплатой: топ-10, рейтинг 2024 — ТОП-10 Рейтинги на DTF', type='url_citation', url='https://dtf.ru/topraiting/3139261-luchshie-fitnes-kluby-moskvy-s-ezhemesyachn

In [51]:
print(f"Ответ без поиска: {len(res.output)}") # напечатает 1
print(f"Ответ с поиском: {len(res_ws.output)}") # напечатает 2

Ответ без поиска: 1
Ответ с поиском: 2


In [52]:
print(res_ws.output[0].action.query)

стоимость годового абонемента в фитнес-клуб в Москве 2024


In [55]:
for x in res_ws.output[-1].content[0].annotations:
    print(f"{x.title} - {x.url}")

Абонемент Максимум + Здоровье - https://spiritfit.ru/maximum-plus-zdorovie/
С чем связан тренд на рекуррентные платежи за фитнес - https://www.kommersant.ru/doc/6877573
Лучшие фитнес-клубы Москвы с ежемесячной оплатой: топ-10, рейтинг 2024 — ТОП-10 Рейтинги на DTF - https://dtf.ru/topraiting/3139261-luchshie-fitnes-kluby-moskvy-s-ezhemesyachnoi-oplatoi-top-10-reiting-2024
Лучшие фитнес-клубы в Москве: ТОП-10 лучших тренажерных залов Москвы с бассейном и без бассейна - https://metaratings.ru/blog/luchshie-fitnes-kluby-v-moskve/
Ежемесячный абонемент по подписке Старт - Абонементы в городе Москва в сети фитнес-залов Spirit.Fitness - https://spiritfit.ru/abonement/tarif-base/


In [56]:
res_ws_2 = client.responses.create(
    model = model,
    previous_response_id = res_ws.id,
    tools = [ { "type": "web_search" } ],
    input = "А поищи самый дешёвый?"
)

printx(res_ws_2.output_text)

 Нашёл варианты самых дешёвых годовых абонементов в фитнес‑клубах Москвы (по оплате в месяц, что выгоднее единовременного годового платежа):

1. **Spirit Fitness** — от 1 700 рублей в месяц. При непрерывной подписке цена фиксируется. Итого за год выходит **20 400 рублей**. В стоимость входит полный безлимит по времени и услугам.

2. **DDX Fitness** — от 1 900 рублей в месяц. В абонемент входит:
* тренажёрный зал;
* групповые занятия;
* личный шкафчик в раздевалке;
* безлимитный доступ.
Итого за год — **22 800 рублей**.

---

**Важные нюансы:**
* Это модели **ежемесячной подписки**, а не разовой оплаты за год. Такой формат часто выгоднее и гибче: если перестаёте ходить, можно приостановить платежи.
* При оформлении может взиматься **разовый вступительный взнос** (уточняйте у клуба).
* Цены актуальны на 2024 год. Рекомендуется уточнять условия на официальных сайтах перед покупкой, так как могут действовать акции или меняться тарифы.

Хотите, проверю конкретные клубы или районы Москвы ещё детальнее?

In [57]:
if len(res_ws_2.output)>1:
    print(res_ws_2.output[0].action.query)
else:
    print("Инструмент поиска не был вызван")

самый дешёвый годовой абонемент в фитнес-клуб в Москве 2024


In [58]:
response = client.responses.create(
    model=model,
    input="Сделай краткий обзор отзывов на World Class Fitness",
    tools=[
        {
            "type": "web_search",
            "filters": {
                "allowed_domains": [
                    "otzovik.com"
                ],
                "user_location": {
                    "region": "213", # Москва
                },
            },
            "search_context_size": "high", # варианты: low | medium | high
        }
    ]
)
printx(response.output_text)

 ### Краткий обзор отзывов на World Class Fitness

**Общая оценка:** 2,3 из 5.  
**Рекомендуют:** 27 % клиентов.  

#### Положительные моменты:
* **Оборудование:** многие отмечают современное и исправное оборудование (в т. ч. тренажёры Technogym).  
* **Чистота:** в ряде отзывов подчёркивается чистота залов, бассейнов и раздевалок.  
* **Бассейны и SPA‑зоны:** клиенты хвалят бассейны, сауны, хамамы.  
* **Тренеры:** часть клиентов выделяет профессионализм тренеров, помощь в составлении программ и корректировке техники.  
* **Расположение:** удобство локации, наличие парковки в некоторых клубах.  
* **Соотношение цены и качества:** отдельные посетители считают его адекватным, особенно при длительных абонементах.  

#### Основные претензии:
* **Высокие цены:** стоимость абонементов и индивидуальных занятий часто называют завышенной, отмечаются резкие повышения цен.  
* **Проблемы с обслуживанием:**  
  * задержки и отказы в возврате средств за неиспользованные абонементы;  
  * длительные сроки выдачи справок для налогового вычета (иногда до полугода), а также ошибки в документах;  
  * непрозрачные условия и системы уровней для клиентов.  
* **Сервис и отношение персонала:**  
  * неклиентоориентированность (в т. ч. грубость и равнодушие);  
  * сложности с получением обратной связи от администрации;  
  * жёсткие условия при осмотре клубов потенциальными клиентами.  
* **Комфорт и инфраструктура:**  
  * переполненность групповых занятий и тренажёрных залов, особенно в вечернее время;  
  * теснота в раздевалках и душевых;  
  * проблемы с вентиляцией и температурой в залах (душно или холодно);  
  * отсутствие бесплатных парковок в ряде локаций.  
* **Безопасность и гигиена:**  
  * случаи нарушения санитарных норм (грязь, запахи);  
  * вопросы к обеспечению безопасности клиентов во время тренировок.  

---

**Вывод:** World Class позиционируется как премиальная сеть фитнес‑клубов, но отзывы демонстрируют значительное расхождение между ожиданиями и реальным опытом. Сильные стороны — оборудование и некоторые SPA‑услуги, однако многочисленные жалобы на сервис, ценовую политику и комфорт снижают общую удовлетворённость клиентов.

In [59]:
response.output

[ResponseFunctionWebSearch(id='3cb9b6fc-95fb-40ad-921f-994b0db2ade9', action=ActionSearch(type='search', queries=None, query='{"query":"отзывы на World Class Fitness","lang":"ru"}', sources=None, valid=True), status='completed', type='web_search_call', valid=True),
 ResponseOutputMessage(id='df663e7e-7cb2-4690-b853-800ef9d65e7b', content=[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=0, start_index=0, title='Отзывы о Фитнес-клуб World Class (Россия, Москва)', type='url_citation', url='https://otzovik.com/reviews/fitnes-klub_world_class_russia_moscow/', valid=True), AnnotationURLCitation(end_index=0, start_index=0, title='Отзыв о Фитнес-клуб World Class (Россия, Москва) | Приходила посмотреть на несколько клубов этой сети. И ни один по-настоящему не понравился.', type='url_citation', url='https://otzovik.com/review_18007494.html', valid=True), AnnotationURLCitation(end_index=0, start_index=0, title='Отзывы о Youtube-канал World Class', type='url_citation', url='https:/

In [60]:
for x in response.output[-1].content[0].annotations:
    print(f"{x.title} - {x.url}")

Отзывы о Фитнес-клуб World Class (Россия, Москва) - https://otzovik.com/reviews/fitnes-klub_world_class_russia_moscow/
Отзыв о Фитнес-клуб World Class (Россия, Москва) | Приходила посмотреть на несколько клубов этой сети. И ни один по-настоящему не понравился. - https://otzovik.com/review_18007494.html
Отзывы о Youtube-канал World Class - https://otzovik.com/reviews/youtube-kanal_world_class/
Отзывы о Сеть фитнесс-клубов World Class - https://otzovik.com/reviews/set_fitness-klubov_world_class/
Отзыв о Фитнес-клуб World Class Lite (Россия, Москва) | Нормальный клуб со своими недостатками,но не для среднего класса. Итак...Девочкины размышления) - https://otzovik.com/review_2511377.html


In [61]:
web_search_tool = {
    "type" : "web_search"
}

instruction = """
Ты — опытный фитнес-тренер, задача которого — помочь мне тренироваться в зале. Ты можешь советовать упражнения, давать рекомендации по питанию и т. д. Отвечай на основе имеющейся дополнительной информации из файловой базы знаний, вызывая инструмент поиска `search_tool`. В случае если запрос касается упоминания абстрактных фитнес-клубов или новостей, используй поиск в интернет `web_search_tool`.
Ты также можешь вести дневник выполненных пользователем упражнений — для этого используй функцию `Exercise`. Чтобы показать список выполненных упражнений, используй `ListExercises`.
"""

fit_agent = Agent(instruction, [web_search_tool, search_tool, Exercise, ListExercises],model=model,tool_choice='required')

res = fit_agent("Сколько стоит годовой абонемент в фитнес-клуб в Санкт-Петербурге?")
printx(res.output_text)

[{'type': 'web_search'}, {'type': 'file_search', 'vector_store_ids': ['fvttid3ao05jp2i343c0'], 'max_num_results': 5}, {'type': 'function', 'name': 'Exercise', 'description': 'Эта функция позволяет добавлять информацию о сделанном в зале упражнении.', 'parameters': {'description': 'Эта функция позволяет добавлять информацию о сделанном в зале упражнении.', 'properties': {'тип': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Тип упражнения (кардио или силовое)', 'title': 'Тип'}, 'название': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Название упражнения', 'title': 'Название'}, 'болевые_ощущения': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Болевые ощущения при выполнении упражнения', 'title': 'Болевые Ощущения'}, 'пульс': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'Пульс в момент выполнения упражнения', 'title': 'Пульс'}, 'подходы': {'a

 Стоимость годового абонемента в фитнес‑клубах Санкт‑Петербурга в 2026 году варьируется в зависимости от клуба и включенных услуг. Вот несколько примеров:

1. **Spirit Fitness** — от 1 700 ₽ в месяц, что составляет **20 400 ₽** за год при непрерывной подписке.
2. **Фитнес‑клуб из рейтинга legospb.com** — 23 990 ₽ за 12 месяцев безлимитного посещения.
3. **Мультикарта** (тренажёрный зал, групповые программы, бассейн и СПА‑зона при наличии в клубе) — 39 000 ₽ за год.
4. **Клуб на Оптиков** — от 16 900 ₽ за тариф «Фитнес‑год».
5. **DDX Fitness** (Чкаловская, открытие 30.10.2026) — от 1 900 ₽ в месяц, то есть **от 22 800 ₽** за год.

**Важные уточнения:**
* В некоторых клубах взимается **вступительный взнос** (например, от 2 500 ₽ при онлайн‑оформлении).
* Цены могут меняться, а акционные тарифы — действовать ограниченное время.
* Стоимость зависит от набора услуг: безлимит, доступ к бассейну, групповым занятиям, СПА и т. д.

Рекомендую уточнять актуальные цены и условия напрямую в интересующем клубе. Хотите, помогу найти информацию по какому‑то конкретному заведению?

## MCP

In [62]:
from fastmcp import FastMCP
from datetime import datetime

mcp = FastMCP("PersonalNotes")

NOTES_BY_ID: dict[int, dict] = {}
NEXT_ID = 1

@mcp.tool(description="Добавить заметку в блокнот")
def add_note(title: str, body: str, notebook: str = "scrapbook") -> dict:
    """Создаёт новую заметку и сохраняет её в блокнот.
    Args:
        title: Заголовок заметки. Не может быть пустым.
        body: Текст заметки. Не может быть пустым.
        notebook: Имя блокнота. Если не указано, используется ``"scrapbook"``.
    """
    global NEXT_ID
    note = {
        "id": NEXT_ID,
        "notebook": notebook,
        "created_at": datetime.now().to_iso_format(),
        "title": title,
        "body": body
    }
    NOTES_BY_ID[NEXT_ID] = note
    NEXT_ID += 1
    return note

ModuleNotFoundError: No module named 'fastmcp'

In [74]:
notes_tool = {
    "type": "mcp",
    "server_label": "PersonalNotes",
    "server_url": "http://158.160.235.239:8000/sse", 
    # Это адрес виртуальной машины
    "require_approval": "never",
}

In [75]:
res = client.responses.create(
    model=model,
    tools=[notes_tool],
    input="Добавь в мой дневник заметку о том, что я сегодня купил хлеб и молоко"
)

In [78]:
res.to_dict()

{'id': '8c2fd046-e3be-423f-a4bf-2ff72bfc4e74',
 'created_at': 1790156578.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt://b1g506hvb02hc6hjcra1/aliceai-llm',
 'object': 'response',
 'output': [{'id': 'f8a2dd02-4f0d-48cf-ad7e-0a5e3dd006b6',
   'server_label': 'PersonalNotes',
   'tools': [{'input_schema': {'type': 'object',
      'properties': {'body': {'description': 'Текст заметки. Не может быть пустым.',
        'type': 'string'},
       'created_at': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
        'default': None,
        'description': 'Дата создания в формате ISO 8601 (``YYYY-MM-DDTHH:MM:SSZ``).\nЕсли не указана, используется текущее время UTC.'},
       'notebook': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
        'default': None,
        'description': 'Имя блокнота. Если не указано, используется ``"scrapbook"``.'},
       'title': {'description': 'Заголовок заметки. Не может быть пустым.',
        'type': 

In [77]:
for x in res.output:
    if x.type=='mcp_call':
        print(f" + Вызов {x.name}{x.arguments}")
        print(f"   Результат: {x.output}")

 + Вызов add_note{"title":"Покупка продуктов","body":"Сегодня купил хлеб и молоко"}
   Результат: {"id":1,"notebook":"scrapbook","created_at":"2026-09-23T09:43:00Z","title":"Покупка продуктов","body":"Сегодня купил хлеб и молоко"}


In [80]:
weather_tool = {
    "type": "mcp",
    "server_label": "weather",
    "server_url": "https://db8qmrktuvcc46b4kfsn.99igvxy3.mcpgw.serverless.yandexcloud.net/sse",
    "require_approval": "never",
}

res = client.responses.create(
    model=model,
    tools=[weather_tool],
    input="Какая погода в Москве?",
)
printx(res.output_text)


 Не удалось получить данные о погоде в Москве из‑за ошибки с API‑ключом. Попробуйте позже или воспользуйтесь другим источником прогноза погоды, например, OpenWeatherMap напрямую.

In [86]:
for x in res.output:
    if x.type=='mcp_call':
        print(f" + Вызов {x.name}{x.arguments}")
        print(f"   Результат: {x.output}")

 + Вызов getweather{"city":"Москва"}
   Результат: None


In [100]:
vkusvil_tool = {
    "type": "mcp",
    "server_label": "vkusvil",
    "server_url": "https://db80epho01h0cqjlatj7.fi4781wp.mcpgw.serverless.yandexcloud.net/sse",
    "require_approval": "never",
}

res = client.responses.create(
    model=model,
    tools=[vkusvil_tool],
    #previous_response_id = res.id,
    input="Почем щечки и есть ли в наличии? на Замшина в СПб",
)
printx(res.output_text)


 В магазинах ВкусВилл в Санкт‑Петербурге есть следующие виды щёчек:

1. **«Щечка телячья томлёная»**
* Цена: 637 руб. за штуку.
* Рейтинг: 4.8 (2 066 оценок).
* Пищевая ценность на 100 г: белки 16 г, жиры 5.6 г, углеводы 0.6 г, соль 0.7 г; 116.8 ккал.
* Состав: щёковина телячья, экстракты специй (перец чёрный, чеснок, кориандр), соль пищевая.
* [Страница товара](https://vkusvill.ru/goods/shchechka-telyachya-tomlenaya-47835/)

2. **«Щёчки говяжьи с сушёными томатами»**
* Цена: 553 руб. за штуку.
* Новинка, пока без рейтинга (4 оценки).
* Пищевая ценность на 100 г: белки 8.4 г, жиры 10.8 г, углеводы 1.3 г, соль 0.6 г; 136 ккал.
* Состав: щёковина говяжья, вода питьевая, масло подсолнечное рафинированное дезодорированное, овощи сушёные (томаты, лук репчатый, петрушка), растительные волокна (цитрусовые), сахар, соль пищевая, экстракты пряностей (паприка, петрушка, кориандр).
* [Страница товара](https://vkusvill.ru/goods/shchechki-govyazhi-s-sushenymi-tomatami-120891/)

К сожалению, не удалось найти магазин ВкусВилл на улице Замшина в Санкт‑Петербурге. Вы можете проверить наличие товара в ближайших магазинах сети или воспользоваться доставкой.

Если хотите, я могу уточнить какую‑либо дополнительную информацию по этим товарам или магазинам!

In [96]:
res

Response(id='3349a753-67aa-4223-a35b-80fed46a8bfe', created_at=1790159450.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt://b1g506hvb02hc6hjcra1/aliceai-llm', object='response', output=[ResponseOutputMessage(id='db88b738-6170-4805-9e4d-cd484691ef05', content=[ResponseOutputText(annotations=[], text=' \n[vkusvil_vkusvill_products_search]\n{"q":"щечки","vvonly":1}\n\n[vkusvil_vkusvill_shops]\n{"id_city_filter":78,"page":1}', type='output_text', logprobs=None, valid=True)], role='assistant', status='completed', type='message', phase=None, valid=True)], parallel_tool_calls=False, temperature=1.0, tool_choice='auto', tools=[Mcp(server_label='vkusvil', type='mcp', allowed_callers=None, allowed_tools=None, authorization=None, connector_id=None, defer_loading=None, headers=None, require_approval='never', server_description=None, server_url='https://db80epho01h0cqjlatj7.fi4781wp.mcpgw.serverless.yandexcloud.net/sse', tunnel_id=None, valid=True)], top_p=1.0, ba

In [99]:
for x in res.output:
    if x.type=='mcp_call':
        print(f" + Вызов {x.name}{x.arguments}")
        print(f"   Результат: {x.output}")